# 03 — Cnn Standardization / 卷积神经网络

**Chapter 24 — File 3 of 6 / 第24章 — 第3个文件（共6个）**

---

## Summary / 总结

This script demonstrates **cnn model with standardization for the har dataset**.

本脚本演示 **cnn model with standardization for the har dataset**。

---
## Step 1 — cnn model with standardization for the har dataset

In [ ]:
from numpy import mean
from numpy import std
from numpy import dstack
from pandas import read_csv
from matplotlib import pyplot
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers import Dropout
from keras.layers.convolutional import Conv1D
from keras.layers.convolutional import MaxPooling1D
from keras.utils import to_categorical

---
## Step 2 — load a single file as a numpy array

In [ ]:
def load_file(filepath):
	dataframe = read_csv(filepath, header=None, delim_whitespace=True)
	return dataframe.values

---
## Step 3 — load a list of files and return as a 3d numpy array

In [ ]:
def load_group(filenames, prefix=''):
	loaded = list()
	for name in filenames:
		data = load_file(prefix + name)
		loaded.append(data)

---
## Step 4 — stack group so that features are the 3rd dimension

In [ ]:
loaded = dstack(loaded)
	return loaded

---
## Step 5 — load a dataset group, such as train or test

In [ ]:
def load_dataset_group(group, prefix=''):
	filepath = prefix + group + '/Inertial Signals/'

---
## Step 6 — load all 9 files as a single array

In [ ]:
filenames = list()

---
## Step 7 — total acceleration

In [ ]:
filenames += ['total_acc_x_'+group+'.txt', 'total_acc_y_'+group+'.txt', 'total_acc_z_'+group+'.txt']

---
## Step 8 — body acceleration

In [ ]:
filenames += ['body_acc_x_'+group+'.txt', 'body_acc_y_'+group+'.txt', 'body_acc_z_'+group+'.txt']

---
## Step 9 — body gyroscope

In [ ]:
filenames += ['body_gyro_x_'+group+'.txt', 'body_gyro_y_'+group+'.txt', 'body_gyro_z_'+group+'.txt']

---
## Step 10 — load input data

In [ ]:
X = load_group(filenames, filepath)

---
## Step 11 — load class output

In [ ]:
y = load_file(prefix + group + '/y_'+group+'.txt')
	return X, y

---
## Step 12 — load the dataset, returns train and test X and y elements

In [ ]:
def load_dataset(prefix=''):

---
## Step 13 — load all train

In [ ]:
trainX, trainy = load_dataset_group('train', prefix + 'HARDataset/')

---
## Step 14 — load all test

In [ ]:
testX, testy = load_dataset_group('test', prefix + 'HARDataset/')

---
## Step 15 — zero-offset class values

In [ ]:
trainy = trainy - 1
	testy = testy - 1

---
## Step 16 — one hot encode y

In [ ]:
trainy = to_categorical(trainy)
	testy = to_categorical(testy)
	return trainX, trainy, testX, testy

---
## Step 17 — standardize data

In [ ]:
def scale_data(trainX, testX, standardize):

---
## Step 18 — remove overlap

In [ ]:
cut = int(trainX.shape[1] / 2)
	longX = trainX[:, -cut:, :]

---
## Step 19 — flatten windows

In [ ]:
longX = longX.reshape((longX.shape[0] * longX.shape[1], longX.shape[2]))

---
## Step 20 — flatten train and test

In [ ]:
flatTrainX = trainX.reshape((trainX.shape[0] * trainX.shape[1], trainX.shape[2]))
	flatTestX = testX.reshape((testX.shape[0] * testX.shape[1], testX.shape[2]))

---
## Step 21 — standardize

In [ ]:
if standardize:
		s = StandardScaler()

---
## Step 22 — fit on training data

In [ ]:
s.fit(longX)

---
## Step 23 — apply to training and test data

In [ ]:
longX = s.transform(longX)
		flatTrainX = s.transform(flatTrainX)
		flatTestX = s.transform(flatTestX)

---
## Step 24 — reshape

In [ ]:
flatTrainX = flatTrainX.reshape((trainX.shape))
	flatTestX = flatTestX.reshape((testX.shape))
	return flatTrainX, flatTestX

---
## Step 25 — fit and evaluate a model

In [ ]:
def evaluate_model(trainX, trainy, testX, testy, param):
	verbose, epochs, batch_size = 0, 10, 32
	n_timesteps, n_features, n_outputs = trainX.shape[1], trainX.shape[2], trainy.shape[1]

---
## Step 26 — scale data

In [ ]:
trainX, testX = scale_data(trainX, testX, param)
	model = Sequential()
	model.add(Conv1D(64, 3, activation='relu', input_shape=(n_timesteps,n_features)))
	model.add(Conv1D(64, 3, activation='relu'))
	model.add(Dropout(0.5))
	model.add(MaxPooling1D())
	model.add(Flatten())
	model.add(Dense(100, activation='relu'))
	model.add(Dense(n_outputs, activation='softmax'))
	model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

---
## Step 27 — fit network

In [ ]:
model.fit(trainX, trainy, epochs=epochs, batch_size=batch_size, verbose=verbose)

---
## Step 28 — evaluate model

In [ ]:
_, accuracy = model.evaluate(testX, testy, batch_size=batch_size, verbose=0)
	return accuracy

---
## Step 29 — summarize scores

In [ ]:
def summarize_results(scores, params):
	print(scores, params)

---
## Step 30 — summarize mean and standard deviation

In [ ]:
for i in range(len(scores)):
		m, s = mean(scores[i]), std(scores[i])
		print('Param=%s: %.3f%% (+/-%.3f)' % (params[i], m, s))

---
## Step 31 — boxplot of scores

In [ ]:
pyplot.boxplot(scores, labels=params)
	pyplot.savefig('exp_cnn_standardize.png')

---
## Step 32 — run an experiment

In [ ]:
def run_experiment(params, repeats=10):

---
## Step 33 — load data

In [ ]:
trainX, trainy, testX, testy = load_dataset()

---
## Step 34 — test each parameter

In [ ]:
all_scores = list()
	for p in params:

---
## Step 35 — repeat experiment

In [ ]:
scores = list()
		for r in range(repeats):
			score = evaluate_model(trainX, trainy, testX, testy, p)
			score = score * 100.0
			print('>p=%s #%d: %.3f' % (p, r+1, score))
			scores.append(score)
		all_scores.append(scores)

---
## Step 36 — summarize results

In [ ]:
summarize_results(all_scores, params)

---
## Step 37 — run the experiment

In [ ]:
n_params = [False, True]
run_experiment(n_params)

---
## Learning Notes / 学习笔记

- **概念**: cnn model with standardization for the har dataset 是机器学习中的常用技术。  
  *cnn model with standardization for the har dataset is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Cnn Standardization / 卷积神经网络
# Complete Code / 完整代码
# ===============================

# cnn model with standardization for the har dataset
from numpy import mean
from numpy import std
from numpy import dstack
from pandas import read_csv
from matplotlib import pyplot
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers import Dropout
from keras.layers.convolutional import Conv1D
from keras.layers.convolutional import MaxPooling1D
from keras.utils import to_categorical

# load a single file as a numpy array
def load_file(filepath):
	dataframe = read_csv(filepath, header=None, delim_whitespace=True)
	return dataframe.values

# load a list of files and return as a 3d numpy array
def load_group(filenames, prefix=''):
	loaded = list()
	for name in filenames:
		data = load_file(prefix + name)
		loaded.append(data)
	# stack group so that features are the 3rd dimension
	loaded = dstack(loaded)
	return loaded

# load a dataset group, such as train or test
def load_dataset_group(group, prefix=''):
	filepath = prefix + group + '/Inertial Signals/'
	# load all 9 files as a single array
	filenames = list()
	# total acceleration
	filenames += ['total_acc_x_'+group+'.txt', 'total_acc_y_'+group+'.txt', 'total_acc_z_'+group+'.txt']
	# body acceleration
	filenames += ['body_acc_x_'+group+'.txt', 'body_acc_y_'+group+'.txt', 'body_acc_z_'+group+'.txt']
	# body gyroscope
	filenames += ['body_gyro_x_'+group+'.txt', 'body_gyro_y_'+group+'.txt', 'body_gyro_z_'+group+'.txt']
	# load input data
	X = load_group(filenames, filepath)
	# load class output
	y = load_file(prefix + group + '/y_'+group+'.txt')
	return X, y

# load the dataset, returns train and test X and y elements
def load_dataset(prefix=''):
	# load all train
	trainX, trainy = load_dataset_group('train', prefix + 'HARDataset/')
	# load all test
	testX, testy = load_dataset_group('test', prefix + 'HARDataset/')
	# zero-offset class values
	trainy = trainy - 1
	testy = testy - 1
	# one hot encode y
	trainy = to_categorical(trainy)
	testy = to_categorical(testy)
	return trainX, trainy, testX, testy

# standardize data
def scale_data(trainX, testX, standardize):
	# remove overlap
	cut = int(trainX.shape[1] / 2)
	longX = trainX[:, -cut:, :]
	# flatten windows
	longX = longX.reshape((longX.shape[0] * longX.shape[1], longX.shape[2]))
	# flatten train and test
	flatTrainX = trainX.reshape((trainX.shape[0] * trainX.shape[1], trainX.shape[2]))
	flatTestX = testX.reshape((testX.shape[0] * testX.shape[1], testX.shape[2]))
	# standardize
	if standardize:
		s = StandardScaler()
		# fit on training data
		s.fit(longX)
		# apply to training and test data
		longX = s.transform(longX)
		flatTrainX = s.transform(flatTrainX)
		flatTestX = s.transform(flatTestX)
	# reshape
	flatTrainX = flatTrainX.reshape((trainX.shape))
	flatTestX = flatTestX.reshape((testX.shape))
	return flatTrainX, flatTestX

# fit and evaluate a model
def evaluate_model(trainX, trainy, testX, testy, param):
	verbose, epochs, batch_size = 0, 10, 32
	n_timesteps, n_features, n_outputs = trainX.shape[1], trainX.shape[2], trainy.shape[1]
	# scale data
	trainX, testX = scale_data(trainX, testX, param)
	model = Sequential()
	model.add(Conv1D(64, 3, activation='relu', input_shape=(n_timesteps,n_features)))
	model.add(Conv1D(64, 3, activation='relu'))
	model.add(Dropout(0.5))
	model.add(MaxPooling1D())
	model.add(Flatten())
	model.add(Dense(100, activation='relu'))
	model.add(Dense(n_outputs, activation='softmax'))
	model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
	# fit network
	model.fit(trainX, trainy, epochs=epochs, batch_size=batch_size, verbose=verbose)
	# evaluate model
	_, accuracy = model.evaluate(testX, testy, batch_size=batch_size, verbose=0)
	return accuracy

# summarize scores
def summarize_results(scores, params):
	print(scores, params)
	# summarize mean and standard deviation
	for i in range(len(scores)):
		m, s = mean(scores[i]), std(scores[i])
		print('Param=%s: %.3f%% (+/-%.3f)' % (params[i], m, s))
	# boxplot of scores
	pyplot.boxplot(scores, labels=params)
	pyplot.savefig('exp_cnn_standardize.png')

# run an experiment
def run_experiment(params, repeats=10):
	# load data
	trainX, trainy, testX, testy = load_dataset()
	# test each parameter
	all_scores = list()
	for p in params:
		# repeat experiment
		scores = list()
		for r in range(repeats):
			score = evaluate_model(trainX, trainy, testX, testy, p)
			score = score * 100.0
			print('>p=%s #%d: %.3f' % (p, r+1, score))
			scores.append(score)
		all_scores.append(scores)
	# summarize results
	summarize_results(all_scores, params)

# run the experiment
n_params = [False, True]
run_experiment(n_params)

---

➡️ **Next / 下一步**: File 4 of 6